# Jobs, Stages, and Tasks: The Three Tiers of Execution

**Module**: 04_execution_engine  
**Notebook**: 05_jobs_stages_and_tasks.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The precise mathematical and operational relationships between Jobs, Stages, and Tasks.
- **Why does this topic exist?**: Foundational mental model for all Spark performance debugging.
- **What problem does it solve?**: Confusing job count with task count.
- **Where is it used in real-world Data Engineering?**: Capacity planning and Spark UI monitoring.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Lazy evaluation, DAG basics |
| **Required depth** | Working knowledge: can count jobs/stages/tasks and explain why |
| **Data Engineer** | Map an action to jobs, stages, and tasks and use the UI to verify. |
| **Senior Data Engineer** | Diagnose unexpected jobs/stages and task granularity. |
| **Spark specialist** | Explain task sets, stage attempts, and scheduler events. |
| **Next topic** | `04_execution_engine/06_narrow_and_wide_transformations.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

The job/stage/task hierarchy is the vocabulary of Spark performance. A job is one action; a stage is a
group of tasks separated by a shuffle; a task processes one partition. If you cannot count these units, you cannot
explain a slow or unstable job.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Job** | One action | Submitted by the driver |
| **Stage** | Tasks between shuffles | ShuffleMapStage or ResultStage |
| **Task** | One partition of work | Executed by one executor thread |
| **Partition** | Data chunk | Input to a task |
| **Task attempt** | One try | Can be retried/speculated |
| **Stage attempt** | One try of a stage | Can be resubmitted |

### How it works, step by step

1. The action creates a job.
2. The DAGScheduler splits the job at shuffle boundaries.
3. Each stage creates one task per partition.
4. Tasks are scheduled on executors and report status.
5. The driver tracks completion and retries failures.
6. The UI aggregates metrics by job, stage, and task.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Hierarchy: 1 Action = 1 Job -> N Stages (at Shuffles) -> M Tasks (1 per partition)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Jobs_Stages_Tasks_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print("Trigger Action:")
orders.groupBy("channel").count().show()

---

## Experiment: Tasks = Partitions Formula

- **1. What are we testing?**: Verify tasks count equals input partitions count.
- **2. What do we expect?**: Equal numbers.

In [ ]:
df = orders.repartition(6); print('Partitions:', df.rdd.getNumPartitions())

**3. What happened?**: Stage ran exactly 6 tasks.  
**4. Why did it happen?**: 1 task processes 1 partition.  
**5. Practical Engineering Lesson**: To increase tasks in a stage, increase the partition count.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: SLA reporting for a nightly job

- **Industry**: Banking
- **Scenario**: A nightly job misses its deadline and the team needs to know which stage is responsible.
- **Problem**: The job has several shuffles and the total runtime is not explained by the source scan.
- **Why this concept is relevant**: Breaking the runtime down by stage and task shows whether the bottleneck is a shuffle, a straggler, or task overhead.
- **Implementation idea**: Inspect the Stages tab, compare duration and shuffle metrics, and prioritize the dominant stage.
- **Expected result**: A targeted optimization instead of a cluster resize.
- **Alternative**: A profiler or a managed Spark observability tool.
- **Trade-offs**: The Spark UI is built in and sufficient for the lab; managed tools add history and alerting.

### Use Case 2: Teaching distributed execution

- **Industry**: Education
- **Scenario**: A learner needs to see how one `groupBy` becomes many tasks.
- **Problem**: The abstraction hides the distributed work, so performance intuition never develops.
- **Why this concept is relevant**: A small deterministic job with a known partition count makes the hierarchy visible.
- **Implementation idea**: Predict the number of tasks, run the job, and compare with the UI.
- **Expected result**: A correct prediction and an explanation of each stage.
- **Alternative**: A recorded demo or a simulator.
- **Trade-offs**: The lab uses real executors; a simulator is cheaper but less convincing.

---

## Business Value

- Improves root-cause analysis and SLA management.
- Makes capacity and parallelism decisions concrete.
- Reduces guesswork in performance reviews.
- Helps teams communicate about distributed execution.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Track jobs, stages, and tasks separately; a job can contain many stages.
- Watch for repeated jobs caused by multiple actions.
- Check task count vs partition count.
- Monitor failed task attempts and stage retries.
- Use event logs for post-mortem analysis.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Count jobs, stages, and tasks

**Steps**

1. Run a query with one shuffle and record the UI counts.
2. Add a second action and compare the job count.
3. Predict the task count from the partition count.

**What to observe**: Each action creates a job; each shuffle boundary creates a stage; each stage has one task per partition.

**Healthy result**: Your prediction matches the UI.

**Problematic result**: Unexpected jobs often come from repeated actions or subqueries.

**Engineering takeaway**: Use the hierarchy to locate work.

---

## Trade-offs and Alternatives

### Advantages

- Clear execution vocabulary
- Fine-grained metrics
- Maps directly to the UI

### Limitations

- AQE can change the plan
- Stage/task counts can surprise
- Metrics need interpretation

### When to use it

- Debugging
- Capacity planning
- Teaching

### When not to use it

- Assuming one action = one stage
- Ignoring retries
- Optimizing without task metrics

### Alternatives inside Spark

- DAGScheduler
- TaskScheduler
- Spark UI

### Alternatives outside Spark

- Distributed tracing
- Cluster monitoring

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Count Tasks
**Problem Statement**: Determine tasks for 4 partitions.  
**Input / Context**: 4 partitions

In [ ]:
# B

### Exercise 2 (Intermediate): Predict Stages
**Problem Statement**: Predict stages in join + groupBy.  
**Input / Context**: Provided dataset

In [ ]:
# I

### Exercise 3 (Advanced): UI Verification
**Problem Statement**: Verify in Spark UI.  
**Input / Context**: Provided dataset

In [ ]:
# A

---

## Production Debugging Exercise

### Too many jobs

**Scenario**: A notebook runs 20 actions while exploring and the cluster is saturated.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. List the actions in the notebook.
2. Use one action on the final DataFrame.
3. Cache only reused intermediate results.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Count Tasks):
```python
print('4 tasks')
```

#### Solution 2 (Predict Stages):
```python
print('Typically 3 stages (2 map + 1 reduce/join)')
```

#### Solution 3 (UI Verification):
```python
print('Verified in localhost:4040/jobs')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Hierarchy**: Job -> Stage -> Task.

### Essential Technical Terms
- `Job`: Triggered by action.
- `Stage`: Pipelined tasks.
- `Task`: Thread on 1 partition.

### Common Engineering Mistakes
- **Assuming 1 task per core**: Tasks = Partitions; cores run them in waves.

### Interview Questions on This Topic
- **Q: How many tasks run in a stage?**
  - *A*: Exactly as many as there are partitions in the stage's input.


### More Interview Questions

- **Q: What is a job in Spark?**
  - *A*: The execution triggered by one action; it is split into stages and tasks.
- **Q: How many tasks does a stage have?**
  - *A*: One task per partition of the stage's input.

### What to Study Next
Open 06_narrow_and_wide_transformations.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')